# 6. Inference and deployment

Un modelo desplegable debe recibir observaciones con el esquema original y aplicar internamente las mismas transformaciones utilizadas durante el entrenamiento.

Este notebook simula dos escenarios:

1. predicción por lotes;
2. función de predicción reutilizable por una API o aplicación.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd "/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification"

/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification


In [3]:
import json
import joblib
import pandas as pd

In [9]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"
for directory in (DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [5]:
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
model = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")
feature_columns = contract["features"]
class_names = contract["target_names"]
print("Modelo cargado con", len(feature_columns), "variables de entrada.")

Modelo cargado con 30 variables de entrada.


## Contrato de inferencia

In [6]:
def validate_input(records: pd.DataFrame) -> pd.DataFrame:
    missing = sorted(set(feature_columns) - set(records.columns))
    extra = sorted(set(records.columns) - set(feature_columns))
    if missing:
        raise ValueError(f"Faltan variables requeridas: {missing}")
    if extra:
        print(f"Advertencia: se ignorarán columnas adicionales: {extra}")
    return records[feature_columns]


def predict(records: pd.DataFrame) -> pd.DataFrame:
    X = validate_input(records.copy())
    labels = model.predict(X)
    probabilities = model.predict_proba(X)
    return pd.DataFrame({
        "prediction": labels,
        "predicted_class": [class_names[label] for label in labels],
        "probability_malignant": probabilities[:, 0],
        "probability_benign": probabilities[:, 1],
    }, index=records.index)

## Predicción por lotes

In [7]:
# En producción este archivo representaría observaciones nuevas, no el test set.
sample = pd.read_csv(DATA_DIR / "train.csv").drop(columns=contract["target"]).head(5)
sample.to_csv(DATA_DIR / "inference_sample.csv", index=False)

predictions = predict(sample)
predictions

,prediction,predicted_class,probability_malignant,probability_benign
0,1,benign,0.000049,0.999951
1,0,malignant,0.999994,0.000006
2,1,benign,0.000006,0.999994
3,1,benign,0.009902,0.990098
4,1,benign,0.000771,0.999229


In [8]:
predictions.to_csv(REPORTS_DIR / "batch_predictions.csv", index=False)
print("Predicciones guardadas.")

Predicciones guardadas.


## Siguiente paso: servir el modelo

La función `predict` contiene el núcleo de inferencia. Una API con FastAPI podría:

1. recibir un JSON;
2. convertirlo en `DataFrame`;
3. validar el contrato;
4. llamar a `predict`;
5. devolver la predicción como JSON.

El artefacto `champion_model.joblib` ya contiene preprocesamiento y clasificador. Para un despliegue real todavía habría que versionar el artefacto, fijar las dependencias, crear pruebas y monitorizar los datos de entrada y las predicciones.